In [2]:
import polars as pl
import numpy as np
from gensim.models import KeyedVectors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pl.read_csv("fyp.csv")

df = df.with_columns([
    pl.col("title").fill_null(""),
    pl.col("abstract").fill_null("")
])

df = df.with_columns(
    (pl.col("title") + " " + pl.col("abstract")).alias("search_text")
)

print(f" Data loaded successfully! Total documents: {len(df)}")

 Data loaded successfully! Total documents: 5341


In [3]:
print("Loading Word2Vec model... Please wait.")
model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)

tfidf_vec = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf_vec.fit_transform(df['search_text'])
word_to_idx = tfidf_vec.vocabulary_

print("Word2Vec model and TF-IDF weights are ready!")

Loading Word2Vec model... Please wait.
Word2Vec model and TF-IDF weights are ready!


In [4]:
def get_weighted_vector(text, row_idx):
    words = str(text).lower().split()
    vectors = []
    weights = []

    for word in words:
        if word in model and word in word_to_idx:
            w_idx = word_to_idx[word]
            weight = tfidf_matrix[row_idx, w_idx]

            vectors.append(model[word] * weight)
            weights.append(weight)

    if not vectors or sum(weights) == 0:
        return np.zeros(300)

    return np.sum(vectors, axis=0) / np.sum(weights)

print("Calculating weighted vectors for all documents...")
doc_vectors = np.array([
    get_weighted_vector(text, i) for i, text in enumerate(df['search_text'])
])
print(f"Vectorization complete! Matrix shape: {doc_vectors.shape}")

Calculating weighted vectors for all documents...
Vectorization complete! Matrix shape: (5341, 300)


In [5]:
def search_engine(query, top_k=5):
    query_words = query.lower().split()
    q_vecs = [model[w] for w in query_words if w in model]

    if not q_vecs:
        return "Sorry, no semantic matches found."

    query_vector = np.mean(q_vecs, axis=0).reshape(1, -1)

    scores = cosine_similarity(query_vector, doc_vectors)[0]

    results = df.with_columns(
        pl.Series("similarity_score", scores)
    ).sort("similarity_score", descending=True)

    print(f"🔍 Search Results for: '{query}'")
    return results.head(top_k)[["title", "similarity_score"]]

search_engine("Deep learning and computer vision")

🔍 Search Results for: 'Deep learning and computer vision'


title,similarity_score
str,f64
"""E-Learning on Movement Control…",0.67664
"""Kids English Learning Applicat…",0.646863
"""Ontology-Based Lecture Topic M…",0.63325
"""Object Recognition with Deep L…",0.614067
"""Effect of Firm Size on Applica…",0.5966
